# hexaload — eco2mix data exploration

Exploration notebook for the eco2mix national cache in `../data/cache/`
(both feeds: `eco2mix-national-cons-def` up to 2026-08-31, then
`eco2mix-national-tr` real-time tail — stitched by the backtest).


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

import hexaload.viz  # noqa: F401 — applies the hexaload plotly template

pio.renderers.default = "notebook"  # plotly.js inline: renders offline

frames = []
for source, cache in [
    ("cons-def", Path("../data/cache/eco2mix-national-cons-def")),
    ("tr", Path("../data/cache/eco2mix-national-tr")),
]:
    files = sorted(cache.glob("*.parquet"))
    if files:
        part = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
        part["source"] = source
        frames.append(part)
        print(f"{source}: {len(files)} monthly files | {len(part):,} records")
df = pd.concat(frames, ignore_index=True)
df["date_heure"] = pd.to_datetime(df["date_heure"], utc=True)
print(
    f"total: {len(df):,} records | period: "
    f"{df['date_heure'].min()} -> {df['date_heure'].max()}"
)
df.head(3)

## 0. TableReport (skrub) — overview of the raw eco2mix data

Interactive HTML report on a 200k-row sample, also saved to
`../data/probe/tablereport.html` (gitignored).

In [ ]:
from skrub import TableReport

sample = df.sample(n=min(200_000, len(df)), random_state=42)
report = TableReport(
    sample,
    title="eco2mix national raw (sample)",
    compute_associations=True,
    plot_distributions=True,
)
Path("../data/probe").mkdir(exist_ok=True)
Path("../data/probe/tablereport.html").write_text(report.html())
report  # inline render in the notebook

## 1. Consumption series + RTE D-1 forecast (France, hourly UTC)

In [ ]:
NATURES = ("Données définitives", "Données consolidées", "Données temps réel")
sub = (
    df[(df["perimetre"] == "France") & (df["nature"].isin(NATURES))]
    .set_index("date_heure")
    .sort_index()
)
hourly = sub.resample("1h")[["consommation", "prevision_j1", "prevision_j"]].mean()
print(f"hourly: {hourly.index.min()} -> {hourly.index.max()} ({len(hourly):,} h)")
week = hourly.loc["2023-06-05":"2023-06-12"]
px.line(
    week,
    title="Consumption vs D-1/D forecasts — June 2023 week (MW, hourly UTC)",
).show()

In [ ]:
h = hourly.dropna(subset=["consommation"]).copy()
h["hour"] = h.index.hour
h["dow"] = h.index.dayofweek
px.line(
    h.groupby("hour")["consommation"].mean(),
    title="Average hourly profile (MW)",
    labels={"hour": "UTC hour", "value": "MW"},
).show()
px.bar(
    h.groupby("dow")["consommation"].mean(),
    title="Average profile by weekday (0=Monday, MW)",
    labels={"dow": "weekday", "value": "MW"},
).show()

In [ ]:
err = (hourly["consommation"] - hourly["prevision_j1"]).dropna()
px.histogram(
    err,
    nbins=120,
    title="D-1 forecast error (consumption − forecast_D-1, MW)",
    labels={"value": "MW"},
).show()
err.describe()

## 2. Validation: is consumption = Σ energy sources?

In eco2mix, `consommation` should equal the sum of production branches
+ `ech_physiques` + `pompage` (pompage is already negative).

In [ ]:
sources = [
    "fioul",
    "charbon",
    "gaz",
    "nucleaire",
    "eolien",
    "solaire",
    "hydraulique",
    "pompage",
    "bioenergies",
    "ech_physiques",
]
missing = [c for c in sources if c not in df.columns]
print("missing source columns:", missing)

# coerce mixed str/int typing; skipna=False -> NaN if any source is missing
df[sources] = df[sources].apply(pd.to_numeric, errors="coerce")
prod_sum = df[sources].sum(axis=1, skipna=False)
gap = df["consommation"] - prod_sum
both = df["consommation"].notna() & prod_sum.notna()
print(f"rows with both consumption and source sum: {both.sum():,}/{len(df):,}")
print(f"|gap| > 1 MW: {(gap[both].abs() > 1).mean():.4%}")
print(gap[both].describe())

px.histogram(
    gap[both],
    nbins=200,
    title="consumption − Σ(sources + ech_physiques + pompage), MW",
    labels={"value": "MW"},
).show()

tmp = df.loc[both, ["date_heure"]].assign(gap=gap[both])
monthly = tmp.set_index("date_heure")["gap"].resample("ME").median()
px.line(monthly, title="Median gap by month (MW)", labels={"value": "MW"}).show()

## 3. Weather — national meteorological variables (archived ICON-EU)

`../data/processed/openmeteo-icon_eu-hourly-national.parquet`: hourly forecasts
from the **ICON-EU** model (DWD, ~7 km grid), taken from the Open-Meteo
**forecast archive** (as-of forecast at the past date, NOT an observation →
leakage-safe), spatially aggregated over a 1° grid of metropolitan France
(154 points; mean for continuous vars, mode for `weather_code`).

Variables: `temperature_2m_c`, `apparent_temperature_c`,
`relative_humidity_2m_pct`, `pressure_msl_hpa`, `precipitation_mm`,
`wind_speed_10m_ms`, `shortwave_radiation_wm2`, `cloud_cover_pct`,
`is_day_fraction`, `weather_code_code`. Real coverage: ~2022-12 → today
(before that, 100 % null in the archive for this model).


In [ ]:
wx_cache = Path("../data/cache/openmeteo-icon_eu")
wx_files = sorted(wx_cache.glob("*.parquet"))
wx_raw = pd.concat((pd.read_parquet(f) for f in wx_files), ignore_index=True)
wx_sample = wx_raw.sample(n=min(200_000, len(wx_raw)), random_state=42)
wx_report = TableReport(wx_sample, title="openmeteo icon_eu raw grid (sample)")
Path("../data/probe").mkdir(exist_ok=True)
Path("../data/probe/tablereport_weather.html").write_text(wx_report.html())
wx_report  # inline render

In [ ]:
wx = pd.read_parquet("../data/processed/openmeteo-icon_eu-hourly-national.parquet")
wx = wx.dropna()
print(f"weather: {len(wx):,} h | {wx.index.min()} -> {wx.index.max()}")
wx.describe().round(2)

In [ ]:
px.line(
    wx.loc["2023-06-05":"2023-06-12"],
    title="National temperature — June 2023 week (°C, hourly UTC)",
).show()
daily = wx["temperature_2m_c"].resample("1D").mean()
px.line(
    daily,
    title="National daily temperature (°C)",
    labels={"value": "°C", "date_heure_utc": "date"},
).show()
doy = wx.assign(doy=wx.index.dayofyear).groupby("doy")["temperature_2m_c"].mean()
px.line(
    doy,
    title="Average seasonal profile (day of year)",
    labels={"doy": "day", "value": "°C"},
).show()

# extra variables appear once the weather cache is refetched (10 vars)
for col, label in [
    ("relative_humidity_2m_pct", "relative humidity (%)"),
    ("pressure_msl_hpa", "sea-level pressure (hPa)"),
    ("precipitation_mm", "precipitation (mm)"),
]:
    if col not in wx.columns:
        print(f"col {col} not present")
        continue
    daily_v = wx[col].resample("1D").mean()
    px.line(daily_v, title=f"National daily {label}", labels={"value": label}).show()

In [ ]:
joined = hourly.join(wx.rename(columns={"temperature_2m_c": "temp_c"}), how="inner")
joined = joined.dropna(subset=["consommation"])
print(f"consumption x weather join: {len(joined):,} common hours")
prof = (
    joined.assign(t=joined["temp_c"].round().astype(int))
    .groupby("t")["consommation"]
    .mean()
)
px.line(
    prof,
    title="Average consumption by temperature (°C) — the U-curve",
    labels={"t": "°C", "value": "MW"},
).show()
px.scatter(
    joined.sample(n=min(20_000, len(joined)), random_state=42),
    x="temp_c",
    y="consommation",
    opacity=0.3,
    title="Consumption vs temperature (20k h sample)",
    labels={"temp_c": "°C", "consommation": "MW"},
).show()

## 4. Conclusions

- 96 records/day (15-min grid); `consommation` is published only on :00/:30
  (30-min effective), `prevision_j1`/`prevision_j` on every quarter-hour.
- Two `nature` values: "Données définitives" (2012→2025-12) and
  "Données consolidées" (2025-12→M-1) — union them for continuity.
- `consommation` = Σ(production branches) + `ech_physiques` + `pompage`,
  exact until 2020-12; from 2021-01 RTE switched to meter readings + flat
  allowances → residual ≈ -0.14 % (~-70 MW), stable by year.
- `ech_physiques` ≠ Σ(`ech_comm_*`): ≈ -232 MW of unlisted interco losses.
- `date`+`heure` = Europe/Paris local; `date_heure` = UTC (DST confirmed).
- Weather archive (icon_eu) is 100 % null before ~2022-12 → usable from
  2023-01. Backtest window: 2023-01 → 2026-11.
- ODRE/RTE API: page limit is 100, `X-RateLimit-Reset` is an epoch timestamp
